In [1]:
pip install monai

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import glob
import torch
import torch.nn as nn
from monai.transforms import (
    Compose, LoadImaged, EnsureChannelFirstd, ScaleIntensityd, CropForegroundd, ToTensord
)
from monai.data import Dataset, DataLoader
from monai.networks.nets import UNet
from monai.losses import DiceLoss
from monai.metrics import DiceMetric

<frozen importlib._bootstrap_external>:1241: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
2025-11-06 16:06:31.251102: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1762445191.273619     228 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1762445191.280655     228 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [3]:
import os, random, numpy as np, torch
from monai.utils import set_determinism

SEED = 42

# Global seeds
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Deterministic behavior
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


# MONAI determinism
set_determinism(seed=SEED)

# Optional: fix environment hash seed


os.environ["PYTHONHASHSEED"] = str(SEED)


In [4]:
import os
import glob

dataset_dir = "/kaggle/input/archive-zip/Mild"

# List files with exact suffixes (no .gz here)
image_files_mild = sorted(glob.glob(os.path.join(dataset_dir, "*_cropped.nii")))
label_files_mild = sorted(glob.glob(os.path.join(dataset_dir, "*_cropped_seg.nii")))

print(f"Found {len(image_files_mild)} images")
print(f"Found {len(label_files_mild)} labels")

# Sanity check first 3 files
print("Sample image files:", image_files_mild[:3])
print("Sample label files:", label_files_mild[:3])

Found 36 images
Found 36 labels
Sample image files: ['/kaggle/input/archive-zip/Mild/pat10_aug1_cropped.nii', '/kaggle/input/archive-zip/Mild/pat10_aug2_cropped.nii', '/kaggle/input/archive-zip/Mild/pat10_cropped.nii']
Sample label files: ['/kaggle/input/archive-zip/Mild/pat10_aug1_cropped_seg.nii', '/kaggle/input/archive-zip/Mild/pat10_aug2_cropped_seg.nii', '/kaggle/input/archive-zip/Mild/pat10_cropped_seg.nii']


In [5]:
import os
import glob

dataset_dir = "/kaggle/input/archive-zip/Moderate"

# List files with exact suffixes (no .gz here)
image_files_moder = sorted(glob.glob(os.path.join(dataset_dir, "*_cropped.nii")))
label_files_moder = sorted(glob.glob(os.path.join(dataset_dir, "*_cropped_seg.nii")))

print(f"Found {len(image_files_moder)} images")
print(f"Found {len(label_files_moder)} labels")

# Sanity check first 3 files
print("Sample image files:", image_files_moder[:3])
print("Sample label files:", label_files_moder[:3])

Found 36 images
Found 36 labels
Sample image files: ['/kaggle/input/archive-zip/Moderate/pat0_aug1_cropped.nii', '/kaggle/input/archive-zip/Moderate/pat0_aug2_cropped.nii', '/kaggle/input/archive-zip/Moderate/pat0_aug3_cropped.nii']
Sample label files: ['/kaggle/input/archive-zip/Moderate/pat0_aug1_cropped_seg.nii', '/kaggle/input/archive-zip/Moderate/pat0_aug2_cropped_seg.nii', '/kaggle/input/archive-zip/Moderate/pat0_aug3_cropped_seg.nii']


In [6]:
import os
import glob

dataset_dir = "/kaggle/input/archive-zip/Severe"

# List files with exact suffixes (no .gz here)
image_files_sever = sorted(glob.glob(os.path.join(dataset_dir, "*_cropped.nii")))
label_files_sever = sorted(glob.glob(os.path.join(dataset_dir, "*_cropped_seg.nii")))

print(f"Found {len(image_files_sever)} images")
print(f"Found {len(label_files_sever)} labels")

# Sanity check first 3 files
print("Sample image files:", image_files_sever[:3])
print("Sample label files:", label_files_sever[:3])

Found 36 images
Found 36 labels
Sample image files: ['/kaggle/input/archive-zip/Severe/pat16_cropped.nii', '/kaggle/input/archive-zip/Severe/pat20_cropped.nii', '/kaggle/input/archive-zip/Severe/pat21_cropped.nii']
Sample label files: ['/kaggle/input/archive-zip/Severe/pat16_cropped_seg.nii', '/kaggle/input/archive-zip/Severe/pat20_cropped_seg.nii', '/kaggle/input/archive-zip/Severe/pat21_cropped_seg.nii']


In [7]:
from monai.transforms import (
    Compose, LoadImaged, EnsureChannelFirstd, ScaleIntensityd,
    ResizeD, CropForegroundd, ToTensord, NormalizeIntensityd,
    # --- Recommended Augmentations ---
    RandFlipd,          # 1. Random Flipping
    RandAffined,        # 2. Random Rotation/Zoom/Shear
    Rand3DElasticd,     # 3. Non-linear Deformation (Highly effective)
    RandGaussianNoised, # 4. Random Noise
    RandAdjustContrastd # 5. Random Contrast/Brightness
)

train_transforms = Compose([
    LoadImaged(keys=["image", "label"]),
    EnsureChannelFirstd(keys=["image", "label"]),
    ScaleIntensityd(keys=["image"]),
    CropForegroundd(keys=["image", "label"], source_key="image"),
    ResizeD(keys=["image"], spatial_size=(96, 96, 96), mode="trilinear"),
    ResizeD(keys=["label"], spatial_size=(96, 96, 96), mode="nearest"),

    # ----- Strong Spatial Augmentations -----
    RandFlipd(keys=["image", "label"], spatial_axis=[0, 1, 2], prob=0.5),

    RandAffined(
        keys=["image", "label"],
        mode=("trilinear", "nearest"),
        prob=0.3,
        rotate_range=(0.1, 0.1, 0.1),      # larger rotations
        scale_range=(0.15, 0.15, 0.15),    # stronger zoom in/out
        shear_range=(0.05, 0.05, 0.05),    # introduce shear
        translate_range=(10, 10, 10),      # random translations
        padding_mode="zeros"
    ),

    Rand3DElasticd(
        keys=["image", "label"],
        sigma_range=(6, 8),
        magnitude_range=(80, 120),         # more aggressive deformation
        prob=0.2,
        rotate_range=(0.05, 0.05, 0.05),
        mode=("trilinear", "nearest")
    ),

    # ----- Strong Intensity Augmentations -----
    RandGaussianNoised(keys="image", prob=0.3, std=0.01),     # more noise
    RandAdjustContrastd(keys="image", prob=0.3, gamma=(0.7, 1.3)),  # bigger contrast changes

    ToTensord(keys=["image", "label"]),
])




val_transforms = Compose([
    LoadImaged(keys=["image", "label"]),
    EnsureChannelFirstd(keys=["image", "label"]),
    ScaleIntensityd(keys=["image"]),   # first scale intensities
    CropForegroundd(keys=["image", "label"], source_key="image"),
    ResizeD(keys=["image"], spatial_size=(96, 96, 96), mode="trilinear"),
    ResizeD(keys=["label"], spatial_size=(96, 96, 96), mode="nearest"),
    ToTensord(keys=["image", "label"]),
])

In [8]:
data_mild = [{"image": img, "label": lbl} for img, lbl in zip(image_files_mild, label_files_mild)]
data_moder = [{"image": img, "label": lbl} for img, lbl in zip(image_files_moder, label_files_moder)]
data_sever = [{"image": img, "label": lbl} for img, lbl in zip(image_files_sever, label_files_sever)]

In [9]:
def worker_init_fn(worker_id):
    np.random.seed(SEED + worker_id)
    random.seed(SEED + worker_id)

In [10]:
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
import numpy as np
from monai.metrics import DiceMetric
from tqdm.notebook import tqdm

dice_metric = DiceMetric(reduction="mean", num_classes=9, include_background=False)

def evaluate_model_1(model, test_loader, loss_function=None, class_idx=1, visualize=False, num_slices=2):
    """
    Evaluate model with Dice metric and optionally visualize MRI, ground truth, prediction, and heatmap.
    Returns mean Dice and mean loss over the test set.

    Args:
        model: trained 3D UNet
        test_loader: DataLoader for validation/test set
        loss_function: torch loss function (optional)
        class_idx: index of class to show probability heatmap
        visualize: if True, shows visualization for first batch
        num_slices: number of slices to visualize (from center)
    Returns:
        mean_loss: average loss over test set (None if loss_function not provided)
        mean_dice: average Dice over test set
    """
    model.eval()
    device = next(model.parameters()).device
    total_loss = 0.0
    steps = 0
    c = 0  # counter for visualization

    with torch.no_grad():
        progress_bar = tqdm(test_loader, desc="Evaluation", leave=False)
        for batch_data in progress_bar:
            steps += 1
            images = batch_data["image"].to(device, non_blocking=True)  # (B, 1, H, W, D)
            labels = batch_data["label"].to(device, non_blocking=True)  # (B, 1, H, W, D)

            outputs = model(images)  # (B, num_classes, H, W, D)
            probs = F.softmax(outputs, dim=1)
            preds = torch.argmax(outputs, dim=1).to(device)

            # Compute loss if loss function is provided
            if loss_function is not None:
                labels_squeezed = labels.long()
                loss = loss_function(outputs, labels_squeezed)
                total_loss += loss.item()

            # Compute Dice
            labels_squeezed = labels.squeeze(1).long() if labels.ndim == 5 else labels.long()
            dice_metric(
                y_pred=F.one_hot(preds, num_classes=9).permute(0,4,1,2,3).float().to(device),
                y=F.one_hot(labels_squeezed, num_classes=9).permute(0,4,1,2,3).float().to(device)
            )

            # Optional visualization for first batch only
            if visualize and c == 0:
                img = images[0, 0].cpu()
                lbl = labels_squeezed[0].cpu()
                pred = preds[0].cpu()
                heatmap = probs[0, class_idx].cpu()

                D = img.shape[2]
                center = D // 2
                half_span = num_slices // 2
                slice_indices = np.arange(center - half_span, center + half_span)

                fig, axs = plt.subplots(num_slices, 4, figsize=(20, 3*num_slices), constrained_layout=True)
                for i, s in enumerate(slice_indices):
                    axs[i, 0].imshow(img[:, :, s], cmap="gray")
                    axs[i, 0].set_title(f"MRI slice {s}"); axs[i, 0].axis("off")

                    axs[i, 1].imshow(lbl[:, :, s], cmap="tab20")
                    axs[i, 1].set_title("Ground Truth"); axs[i, 1].axis("off")

                    axs[i, 2].imshow(pred[:, :, s], cmap="tab20")
                    axs[i, 2].set_title("Prediction"); axs[i, 2].axis("off")

                    im = axs[i, 3].imshow(heatmap[:, :, s], cmap="hot")
                    axs[i, 3].set_title(f"Heatmap (class {class_idx})"); axs[i, 3].axis("off")

                fig.colorbar(im, ax=axs[:, 3], orientation='vertical', fraction=0.02)
                plt.show()
                c += 1

            # Update progress bar with running metrics
            avg_loss = total_loss / steps if loss_function is not None else 0
            running_dice = dice_metric.aggregate().item()
            progress_bar.set_postfix({"Loss": f"{avg_loss:.4f}", "Dice": f"{running_dice:.4f}"})

        mean_dice = dice_metric.aggregate().item()
        dice_metric.reset()
        mean_loss = total_loss / len(test_loader) if loss_function is not None else None

    return mean_loss, mean_dice

In [11]:
import torch
import matplotlib.pyplot as plt
import random

def visualize_batch_slices(images, preds, gts, num_samples=3):
    """
    images: tensor (B,1,H,W)
    preds:  tensor (B,H,W)
    gts:    tensor (B,H,W)
    """

    # Move to CPU if needed
    images = images.detach().cpu()
    preds  = preds.detach().cpu()
    gts    = gts.detach().cpu()

    B = images.shape[0]
    idxs = random.sample(range(B), min(num_samples, B))

    fig, axes = plt.subplots(len(idxs), 3, figsize=(10, 4 * len(idxs)))

    if len(idxs) == 1:
        axes = [axes]  # make it iterable if only 1 sample

    for row, idx in enumerate(idxs):
        img = images[idx, 0]   # (H,W)
        pred = preds[idx]      # (H,W)
        gt = gts[idx]          # (H,W)

        axes[row][0].imshow(img, cmap="gray")
        axes[row][0].set_title(f"Image Slice #{idx}")
        axes[row][0].axis("off")

        axes[row][1].imshow(pred)
        axes[row][1].set_title("Predicted Mask")
        axes[row][1].axis("off")

        axes[row][2].imshow(gt.squeeze(0))
        axes[row][2].set_title("Ground Truth Mask")
        axes[row][2].axis("off")

    plt.tight_layout()
    plt.show()


In [ ]:
import os
import torch
import torch.nn.functional as F
import numpy as np
from monai.losses import DiceCELoss
from monai.metrics import DiceMetric
from monai.networks.nets import UNet
from monai.networks.layers import Norm
from monai.data import CacheDataset
from torch.utils.data import  ConcatDataset, DataLoader
from sklearn.model_selection import KFold
from tqdm.notebook import tqdm

# =============================
# User-defined parameters
# =============================
device = torch.device("cuda:0")
num_classes = 9
k_folds = 5
max_epochs = 300
val_interval = 1
use_amp = True
SEED = 42
torch.manual_seed(SEED)
g = torch.Generator()
g.manual_seed(SEED)
torch.backends.cudnn.benchmark = True  # ✅ enables conv autotuning

# =============================
# K-Fold splits
# =============================
kf = KFold(n_splits=k_folds, shuffle=True, random_state=SEED)
mild_folds = list(kf.split(data_mild))
moder_folds = list(kf.split(data_moder))
sever_folds = list(kf.split(data_sever))

# =============================
# Containers for metrics
# =============================
fold_train_loss, fold_train_dice = [], []
fold_val_loss_mild, fold_val_loss_moder, fold_val_loss_sever = [], [], []
fold_val_dice_mild, fold_val_dice_moder, fold_val_dice_sever = [], [], []

# =============================
# FOLD LOOP
# =============================
for fold in range(k_folds):
    print(f"\n🚀 Starting Fold {fold+1}/{k_folds}")
    train_dice = mild_dice = moder_dice = sever_dice = 0.0

    # --- Split data ---
    mild_train_idx, mild_val_idx = mild_folds[fold]
    moder_train_idx, moder_val_idx = moder_folds[fold]
    sever_train_idx, sever_val_idx = sever_folds[fold]

    mild_train_list = [data_mild[i] for i in mild_train_idx]
    moder_train_list = [data_moder[i] for i in moder_train_idx]
    sever_train_list = [data_sever[i] for i in sever_train_idx]

    mild_val_list = [data_mild[i] for i in mild_val_idx]
    moder_val_list = [data_moder[i] for i in moder_val_idx]
    sever_val_list = [data_sever[i] for i in sever_val_idx]

    # --- ✅ CacheDataset for faster IO ---
    mild_train_ds = CacheDataset(mild_train_list, transform=train_transforms, cache_rate=1.0, num_workers=8)
    moder_train_ds = CacheDataset(moder_train_list, transform=train_transforms, cache_rate=1.0, num_workers=8)
    sever_train_ds = CacheDataset(sever_train_list, transform=train_transforms, cache_rate=1.0, num_workers=8)

    mild_val_ds = CacheDataset(mild_val_list, transform=val_transforms, cache_rate=1.0, num_workers=4)
    moder_val_ds = CacheDataset(moder_val_list, transform=val_transforms, cache_rate=1.0, num_workers=4)
    sever_val_ds = CacheDataset(sever_val_list, transform=val_transforms, cache_rate=1.0, num_workers=4)

    # --- Combine training datasets ---
    train_ds = ConcatDataset([mild_train_ds, moder_train_ds, sever_train_ds])

    # --- ✅ DataLoaders with persistent_workers ---
    train_loader = DataLoader(
        train_ds, batch_size=24, shuffle=True,
        num_workers=4, pin_memory=True, persistent_workers=True,
        worker_init_fn=worker_init_fn, generator=g
    )
    mild_loader = DataLoader(
        mild_val_ds, batch_size=8,
        num_workers=4, pin_memory=True, persistent_workers=True,
        worker_init_fn=worker_init_fn, generator=g
    )
    moder_loader = DataLoader(
        moder_val_ds, batch_size=8,
        num_workers=4, pin_memory=True, persistent_workers=True,
        worker_init_fn=worker_init_fn, generator=g
    )
    sever_loader = DataLoader(
        sever_val_ds, batch_size=8,
        num_workers=4, pin_memory=True, persistent_workers=True,
        worker_init_fn=worker_init_fn, generator=g
    )

    # --- Model, loss, optimizer, scheduler ---
    model = UNet(
        spatial_dims=3, in_channels=1, out_channels=num_classes,
        channels=(16, 32, 64, 128, 256, 512),
        strides=(2, 2, 2, 2, 2),
        num_res_units=2, norm=Norm.BATCH
    ).to(device)

    loss_function = DiceCELoss(to_onehot_y=True, softmax=True, include_background=False, label_smoothing=0.1)
    optimizer = torch.optim.Adam(model.parameters(), lr=5e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.9, patience=5, min_lr=1e-6)
    dice_metric = DiceMetric(reduction="mean", num_classes=num_classes, include_background=False)
    scaler = torch.amp.GradScaler("cuda")  # ✅ new AMP syntax

    # --- Trackers ---
    train_loss_values, train_dice_values = [], []
    mild_dice_f, moder_dice_f, sever_dice_f = [], [], []
    val_loss_mild_list, val_loss_moder_list, val_loss_sever_list = [], [], []
    # =============================
    # EPOCH LOOP
    # =============================
    for epoch in range(max_epochs):
        model.train()
        epoch_loss = 0
        step = 0
        dice_metric.reset()

        progress_bar = tqdm(train_loader, desc=f"Fold {fold+1} | Epoch {epoch+1}/{max_epochs}", leave=False)
        for batch_data in progress_bar:
            step += 1
            inputs = batch_data["image"].to(device, non_blocking=True)
            labels = batch_data["label"].to(device, non_blocking=True)

            optimizer.zero_grad()
            with torch.amp.autocast("cuda", enabled=use_amp):
                outputs = model(inputs)
                loss = loss_function(outputs, labels.long())

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            epoch_loss += loss.item()
            avg_loss = epoch_loss / step
            progress_bar.set_postfix({"Train_Loss": f"{avg_loss:.4f}"})

        epoch_loss /= step
        train_loss_values.append(epoch_loss)

        # --- Training Dice ---
        model.eval()
        dice_metric.reset()
        with torch.no_grad(), torch.amp.autocast("cuda", enabled=use_amp):
            for batch_data in train_loader:
                inputs = batch_data["image"].to(device)
                labels = batch_data["label"].to(device)
                outputs = model(inputs)
                preds = torch.argmax(outputs, dim=1)
                dice_metric(
                    y_pred=F.one_hot(preds, num_classes=num_classes).permute(0,4,1,2,3).float(),
                    y=F.one_hot(labels.squeeze(1).long(), num_classes=num_classes).permute(0,4,1,2,3).float()
                )
        train_dice = dice_metric.aggregate().item()
        train_dice_values.append(train_dice)
        print(f"Epoch [{epoch+1}/{max_epochs}] | Train Dice: {train_dice:.4f}")

        # --- Validation ---
        if (epoch + 1) % val_interval == 0:
            model.eval()
            with torch.no_grad():
                val_loss_mild, mild_dice = evaluate_model_1(model, mild_loader, loss_function)
                val_loss_moder, moder_dice = evaluate_model_1(model, moder_loader, loss_function)
                val_loss_sever, sever_dice = evaluate_model_1(model, sever_loader, loss_function)
                

            avg_val_loss = (val_loss_mild + val_loss_moder + val_loss_sever) / 3
            scheduler.step(avg_val_loss)
            val_loss_mild_list.append(val_loss_mild)
            val_loss_moder_list.append(val_loss_moder)
            val_loss_sever_list.append(val_loss_sever)
            mild_dice_f.append(mild_dice)
            moder_dice_f.append(moder_dice)
            sever_dice_f.append(sever_dice)

            print(f"Validation - Mild: {mild_dice:.4f} | Moder: {moder_dice:.4f} | Sever: {sever_dice:.4f}")

    # --- Store fold metrics ---
    fold_train_loss.append(train_loss_values)
    fold_train_dice.append(train_dice_values)
    fold_val_dice_mild.append(mild_dice_f)
    fold_val_dice_moder.append(moder_dice_f)
    fold_val_dice_sever.append(sever_dice_f)
    print(f"\n📊 Fold {fold+1} Summary")
    print("=" * 60)
    print(f"Train Loss per epoch: {train_loss_values}")
    print(f"Train Dice per epoch: {train_dice_values}")
    print(f"Validation Loss (Mild): {val_loss_mild_list}")
    print(f"Validation Loss (Moder): {val_loss_moder_list}")
    print(f"Validation Loss (Sever): {val_loss_sever_list}")
    print(f"Validation Dice (Mild): {mild_dice_f}")
    print(f"Validation Dice (Moder): {moder_dice_f}")
    print(f"Validation Dice (Sever): {sever_dice_f}")


🚀 Starting Fold 1/5


Loading dataset: 100%|██████████| 8/8 [00:00<00:00, 20.76it/s]


Fold 1 | Epoch 1/300:   0%|          | 0/4 [00:00<?, ?it/s]

Epoch [1/300] | Train Dice: 0.0194


Evaluation:   0%|          | 0/1 [00:00<?, ?it/s]

...

Evaluation:   0%|          | 0/1 [00:00<?, ?it/s]

Evaluation:   0%|          | 0/1 [00:00<?, ?it/s]

Evaluation:   0%|          | 0/1 [00:00<?, ?it/s]

Validation - Mild: 0.8565 | Moder: 0.7649 | Sever: 0.6462


Fold 4 | Epoch 246/300:   0%|          | 0/4 [00:00<?, ?it/s]

In [ ]:
import os
import torch
import torch.nn.functional as F
import numpy as np
from monai.losses import DiceCELoss
from monai.metrics import DiceMetric
from monai.networks.nets import UNet
from monai.networks.layers import Norm
from monai.data import CacheDataset
from torch.utils.data import ConcatDataset, DataLoader
from sklearn.model_selection import KFold
from tqdm.notebook import tqdm

# =============================
# User-defined parameters
# =============================
device = torch.device("cuda:0")
num_classes = 9
k_folds = 5
max_epochs = 300
val_interval = 1
use_amp = True
SEED = 42
torch.manual_seed(SEED)
g = torch.Generator()
g.manual_seed(SEED)
torch.backends.cudnn.benchmark = True
use_amp = True

# =============================
# K-Fold splits
# =============================
kf = KFold(n_splits=k_folds, shuffle=True, random_state=SEED)
mild_folds = list(kf.split(data_mild))
moder_folds = list(kf.split(data_moder))
sever_folds = list(kf.split(data_sever))

# =============================
# Containers for metrics
# =============================
fold_train_loss, fold_train_dice = [], []
fold_val_loss_mild, fold_val_loss_moder, fold_val_loss_sever = [], [], []
fold_val_dice_mild, fold_val_dice_moder, fold_val_dice_sever = [], [], []

# =============================
# FOLD LOOP
# =============================
for fold in range(k_folds):
    print(f"\n🚀 Starting Fold {fold+1}/{k_folds}")
    train_dice = mild_dice = moder_dice = sever_dice = 0.0

    # --- Split data ---
    mild_train_idx, mild_val_idx = mild_folds[fold]
    moder_train_idx, moder_val_idx = moder_folds[fold]
    sever_train_idx, sever_val_idx = sever_folds[fold]

    mild_train_list = [data_mild[i] for i in mild_train_idx]
    moder_train_list = [data_moder[i] for i in moder_train_idx]
    sever_train_list = [data_sever[i] for i in sever_train_idx]

    mild_val_list = [data_mild[i] for i in mild_val_idx]
    moder_val_list = [data_moder[i] for i in moder_val_idx]
    sever_val_list = [data_sever[i] for i in sever_val_idx]

    # --- CacheDataset for faster IO ---
    mild_train_ds = CacheDataset(mild_train_list, transform=train_transforms, cache_rate=1.0, num_workers=8)
    moder_train_ds = CacheDataset(moder_train_list, transform=train_transforms, cache_rate=1.0, num_workers=8)
    sever_train_ds = CacheDataset(sever_train_list, transform=train_transforms, cache_rate=1.0, num_workers=8)

    mild_val_ds = CacheDataset(mild_val_list, transform=val_transforms, cache_rate=1.0, num_workers=4)
    moder_val_ds = CacheDataset(moder_val_list, transform=val_transforms, cache_rate=1.0, num_workers=4)
    sever_val_ds = CacheDataset(sever_val_list, transform=val_transforms, cache_rate=1.0, num_workers=4)

    # --- Combine training datasets ---
    train_ds = ConcatDataset([mild_train_ds, moder_train_ds, sever_train_ds])

    # --- DataLoaders ---
    train_loader = DataLoader(
        train_ds, batch_size=2, shuffle=True,
        num_workers=4, pin_memory=True, persistent_workers=True,
        worker_init_fn=worker_init_fn, generator=g
    )
    mild_loader = DataLoader(
        mild_val_ds, batch_size=2,
        num_workers=4, pin_memory=True, persistent_workers=True,
        worker_init_fn=worker_init_fn, generator=g
    )
    moder_loader = DataLoader(
        moder_val_ds, batch_size=2,
        num_workers=4, pin_memory=True, persistent_workers=True,
        worker_init_fn=worker_init_fn, generator=g
    )
    sever_loader = DataLoader(
        sever_val_ds, batch_size=2,
        num_workers=4, pin_memory=True, persistent_workers=True,
        worker_init_fn=worker_init_fn, generator=g
    )

    # --- Model, loss, optimizer, scheduler ---
    model = UNet_2D(in_channels=1, num_classes=9).to(device)
    loss_function = DiceCELoss(to_onehot_y=True, softmax=True, include_background=False, label_smoothing=0.1)
    optimizer = torch.optim.Adam(model.parameters(), lr=5e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.9, patience=5, min_lr=1e-6)
    dice_metric = DiceMetric(reduction="mean", num_classes=num_classes, include_background=False)
    scaler = torch.amp.GradScaler("cuda")

    # --- Trackers ---
    train_loss_values, train_dice_values = [], []
    mild_dice_f, moder_dice_f, sever_dice_f = [], [], []

    # ✅ NEW: Track validation losses
    val_loss_mild_list, val_loss_moder_list, val_loss_sever_list = [], [], []

    # =============================
    # EPOCH LOOP
    # =============================
    for epoch in range(max_epochs):
        model.train()
        epoch_loss = 0
        step = 0
        dice_metric.reset()

        progress_bar = tqdm(train_loader, desc=f"Fold {fold+1} | Epoch {epoch+1}/{max_epochs}", leave=False)
        for batch_data in progress_bar:
            step += 1
            inputs = batch_data["image"]
            labels = batch_data["label"]
            slicer = SingleVolumeSliceDataset(inputs, labels, mode=1, image_size=96)
            img2d, mask2d = slicer.get_batch()

            img2d = img2d.to(device, non_blocking=True).float()
            mask2d = mask2d.to(device, non_blocking=True).long()

            optimizer.zero_grad()
            with torch.amp.autocast("cuda", enabled=use_amp):
                outputs = model(img2d)
                loss = loss_function(outputs, mask2d.long())

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            epoch_loss += loss.item()
            avg_loss = epoch_loss / step
            progress_bar.set_postfix({"Train_Loss": f"{avg_loss:.4f}"})

        epoch_loss /= step
        train_loss_values.append(epoch_loss)

        # --- Training Dice ---
        model.eval()
        dice_metric.reset()
        progress_bar = tqdm(train_loader, desc=f"Training DICE Fold {fold+1} | Epoch {epoch+1}/{max_epochs}", leave=False)
        with torch.no_grad(), torch.amp.autocast("cuda", enabled=use_amp):
            for batch_data in progress_bar:
                inputs = batch_data["image"]
                labels = batch_data["label"]
                slicer = SingleVolumeSliceDataset(inputs, labels, mode=1, image_size=96)
                img2d, mask2d = slicer.get_batch()
                img2d = img2d.to(device, non_blocking=True).float()
                mask2d = mask2d.to(device, non_blocking=True).long()
                outputs = model(img2d)
                preds = torch.argmax(outputs, dim=1)
                y_pred = F.one_hot(preds, num_classes=num_classes).permute(0,3,1,2).float()
                y_true = F.one_hot(mask2d.squeeze(1), num_classes=num_classes).permute(0,3,1,2).float()
                dice_metric(y_pred=y_pred, y=y_true)
        train_dice = dice_metric.aggregate().item()
        train_dice_values.append(train_dice)
        print(f"Epoch [{epoch+1}/{max_epochs}] | Train Dice: {train_dice:.4f}")

        # --- Validation ---
        if (epoch + 1) % val_interval == 0:
            val_loss_mild, mild_dice = evaluate_model_2d(model, mild_loader, loss_function)
            val_loss_moder, moder_dice = evaluate_model_2d(model, moder_loader, loss_function)
            val_loss_sever, sever_dice = evaluate_model_2d(model, sever_loader, loss_function)

            avg_val_loss = (val_loss_mild + val_loss_moder + val_loss_sever) / 3
            scheduler.step(avg_val_loss)

            # ✅ Store validation losses and dice
            val_loss_mild_list.append(val_loss_mild)
            val_loss_moder_list.append(val_loss_moder)
            val_loss_sever_list.append(val_loss_sever)
            mild_dice_f.append(mild_dice)
            moder_dice_f.append(moder_dice)
            sever_dice_f.append(sever_dice)
            print(f"Validation - Mild: {mild_dice:.4f} | Moder: {moder_dice:.4f} | Sever: {sever_dice:.4f}")

    # --- Store fold metrics ---
    fold_train_loss.append(train_loss_values)
    fold_train_dice.append(train_dice_values)
    fold_val_loss_mild.append(val_loss_mild_list)
    fold_val_loss_moder.append(val_loss_moder_list)
    fold_val_loss_sever.append(val_loss_sever_list)
    fold_val_dice_mild.append(mild_dice_f)
    fold_val_dice_moder.append(moder_dice_f)
    fold_val_dice_sever.append(sever_dice_f)

    # ✅ PRINT FINAL RESULTS AFTER EACH FOLD
    print(f"\n📊 Fold {fold+1} Summary")
    print("=" * 60)
    print(f"Train Loss per epoch: {train_loss_values}")
    print(f"Train Dice per epoch: {train_dice_values}")
    print(f"Validation Loss (Mild): {val_loss_mild_list}")
    print(f"Validation Loss (Moder): {val_loss_moder_list}")
    print(f"Validation Loss (Sever): {val_loss_sever_list}")
    print(f"Validation Dice (Mild): {mild_dice_f}")
    print(f"Validation Dice (Moder): {moder_dice_f}")
    print(f"Validation Dice (Sever): {sever_dice_f}")


In [ ]:
# ✅ PRINT FINAL RESULTS AFTER EACH FOLD
print(f"\n📊 Fold {fold+1} Summary")
print("=" * 60)
print(f"Train Loss per epoch: {train_loss_values}")
print(f"Train Dice per epoch: {train_dice_values}")
print(f"Validation Loss (Mild): {val_loss_mild_list}")
print(f"Validation Loss (Moder): {val_loss_moder_list}")
print(f"Validation Loss (Sever): {val_loss_sever_list}")
print(f"Validation Dice (Mild): {mild_dice_f}")
print(f"Validation Dice (Moder): {moder_dice_f}")
print(f"Validation Dice (Sever): {sever_dice_f}")

In [ ]:
print("=== Training Loss ===")
print("Mean per epoch:", fold_train_loss)
print("SD per epoch:  ", fold_train_loss)

print("\n=== Training Dice ===")
print("Mean per epoch:", fold_train_dice)
print("SD per epoch:  ", fold_train_dice)

print("\n=== Validation Loss ===")
for severity in ["mild", "moder", "sever"]:
    print(f"{severity.capitalize()} Mean per epoch: {val_loss_mean[severity]}")
    print(f"{severity.capitalize()} SD per epoch:   {val_loss_sd[severity]}")

print("\n=== Validation Dice ===")
for severity in ["mild", "moder", "sever"]:
    print(f"{severity.capitalize()} Mean per epoch: {val_dice_mean[severity]}")
    print(f"{severity.capitalize()} SD per epoch:   {val_dice_sd[severity]}")


In [ ]:
print(mild_dice_f)

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import clear_output
import numpy as np

def plot_training_curves(train_loss,
                         val_loss_mild, val_loss_moder, val_loss_sever,
                         train_dice, mild_dice, moder_dice, sever_dice):
    """Plots train/val losses for 3 classes separately and Dice curves together."""
    clear_output(wait=True)
    epochs = len(train_dice)-1
    x = np.arange(1, epochs + 1)

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # ---- Loss curves ----
    axes[0].plot(x, train_loss_values[:-1], label="Train Loss", linewidth=2)
    axes[0].plot(x, val_loss_mild, label="Val Loss (Mild)", linestyle="--")
    axes[0].plot(x, val_loss_moder, label="Val Loss (Moderate)", linestyle="--")
    axes[0].plot(x, val_loss_sever, label="Val Loss (Severe)", linestyle="--")
    axes[0].set_title("Training & Validation Loss per Class", fontsize=14)
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].legend()
    axes[0].grid(True)

    # ---- Dice curves ----
    axes[1].plot(x, train_dice[:-1], label="Train Dice", linewidth=2)
    axes[1].plot(x, mild_dice, label="Val Dice (Mild)", linestyle="--")
    axes[1].plot(x, moder_dice, label="Val Dice (Moderate)", linestyle="--")
    axes[1].plot(x, sever_dice, label="Val Dice (Severe)", linestyle="--")
    axes[1].set_title("Training & Validation Dice", fontsize=14)
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Dice")
    axes[1].legend()
    axes[1].grid(True)

    plt.tight_layout()
    plt.show()
plot_training_curves(train_loss_values, val_loss_mild_list, val_loss_moder_list, val_loss_sever_list,
    train_dice_values, mild_dice_f, moder_dice_f, sever_dice_f
)

In [ ]:
torch.save({
    'epoch': epoch + 1,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'scheduler_state_dict': scheduler.state_dict(),
    'best_metric_epoch': 148,
}, "/kaggle/working/latest_model.pth")
